In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-09-29 13:27:16--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-09-29 13:27:16 (22.4 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    dataset = f.read()



In [ ]:
print(dataset[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
print(len(dataset))

1115394


In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
sample_encode = tokenizer.encode("hi, hello , my name is shivang")

In [ ]:
print(sample_encode)

[5303, 11, 23748, 837, 616, 1438, 318, 427, 452, 648]


In [ ]:
print(tokenizer.decode(sample_encode))

hi, hello , my name is shivang


In [ ]:
import torch

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

Using: cuda


In [ ]:
torch.manual_seed(42)

In [ ]:
tokens = tokenizer.encode(dataset)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
print(tokens[:100])

[5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248, 461, 11, 2740, 13, 198, 198, 5962, 22307, 25, 198, 1639, 389, 477, 12939, 2138, 284, 4656, 621, 284, 1145, 680, 30, 198, 198, 3237, 25, 198, 4965, 5634, 13, 12939, 13, 198, 198, 5962, 22307, 25, 198, 5962, 11, 345, 760, 327, 1872, 385, 1526, 28599, 318, 4039, 4472, 284, 262, 661, 13, 198, 198, 3237, 25, 198, 1135, 760, 470, 11, 356, 760, 470, 13, 198, 198, 5962, 22307, 25, 198, 5756, 514, 1494, 683, 11, 290, 356]


In [ ]:
data = torch.tensor(tokens, dtype=torch.long)

In [ ]:
split = int(0.9 * len(data))

train_tokens = data[:split]
val_tokens = data[split:]

print("Train:", train_tokens.shape)
print("Validation:", val_tokens.shape)

Train: torch.Size([304222])
Validation: torch.Size([33803])


In [ ]:
block_size = 64
batch_size = 16

In [ ]:
def get_batch(data, batch_size, block_size):
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])

    return x, y

In [ ]:
x, y = get_batch(train_tokens, batch_size, block_size)

print("X shape:", x.shape)
print("Y shape:", y.shape)

X shape: torch.Size([16, 64])
Y shape: torch.Size([16, 64])


In [ ]:
import torch
import torch.nn as nn

vocab_size = tokenizer.vocab_size
embedding_dim = 128

token_embedding = nn.Embedding(vocab_size, embedding_dim)

position_embedding = nn.Embedding(block_size, embedding_dim)

token_emb = token_embedding(x)
positions = torch.arange(block_size)
pos_emb = position_embedding(positions)
embeddings = token_emb + pos_emb

print("Token embedding shape:", token_emb.shape)
print("Position embedding shape:", pos_emb.shape)
print("Final embedding shape:", embeddings.shape)

Token embedding shape: torch.Size([16, 64, 128])
Position embedding shape: torch.Size([64, 128])
Final embedding shape: torch.Size([16, 64, 128])


In [ ]:
query_layer = nn.Linear(128, 128)
key_layer = nn.Linear(128, 128)
value_layer = nn.Linear(128, 128)

Q = query_layer(embeddings)
K = key_layer(embeddings)
V = value_layer(embeddings)

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

Q: torch.Size([16, 64, 128])
K: torch.Size([16, 64, 128])
V: torch.Size([16, 64, 128])


In [ ]:
scores = Q @ K.transpose(-2, -1)
print(scores.shape)

torch.Size([16, 64, 64])


In [ ]:
scores = scores / (embedding_dim ** 0.5)
mask = torch.tril(torch.ones(block_size, block_size))

scores = scores.masked_fill(mask == 0, float('-inf'))

attention_weights = torch.softmax(scores, dim=-1)

print("Attention weights:", attention_weights.shape)

context = attention_weights @ V

print("Context:", context.shape)

Attention weights: torch.Size([16, 64, 64])
Context: torch.Size([16, 64, 128])


In [ ]:
num_heads = 4
head_dim = embedding_dim // num_heads

query_layer = nn.Linear(embedding_dim, embedding_dim)
key_layer = nn.Linear(embedding_dim, embedding_dim)
value_layer = nn.Linear(embedding_dim, embedding_dim)

Q = query_layer(embeddings)
K = key_layer(embeddings)
V = value_layer(embeddings)

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

Q: torch.Size([16, 64, 128])
K: torch.Size([16, 64, 128])
V: torch.Size([16, 64, 128])


In [ ]:
Q = Q.view(batch_size, block_size, num_heads, head_dim)
K = K.view(batch_size, block_size, num_heads, head_dim)
V = V.view(batch_size, block_size, num_heads, head_dim)

In [ ]:
Q = Q.transpose(1, 2)
K = K.transpose(1, 2)
V = V.transpose(1, 2)

In [ ]:
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

Q: torch.Size([16, 4, 64, 32])
K: torch.Size([16, 4, 64, 32])
V: torch.Size([16, 4, 64, 32])


In [ ]:
scores = Q @ K.transpose(-2,-1)
print(scores.shape)

torch.Size([16, 4, 64, 64])


In [ ]:
scores = scores / (head_dim ** 0.5)

In [ ]:
mask = torch.tril(torch.ones(block_size, block_size))

scores = scores.masked_fill(mask == 0, float('-inf'))

attention_weights = torch.softmax(scores, dim=-1)

print(attention_weights.shape)

context = attention_weights @ V
print(context.shape)

torch.Size([16, 4, 64, 64])
torch.Size([16, 4, 64, 32])


In [ ]:
context = context.transpose(1,2)
print(context.shape)

torch.Size([16, 64, 4, 32])


In [ ]:
context = context.contiguous().view(
    batch_size,
    block_size,
    embedding_dim
)

print(context.shape)

torch.Size([16, 64, 128])


In [ ]:
output_projection = nn.Linear(embedding_dim , embedding_dim)
output = output_projection(context)
print(output.shape)

torch.Size([16, 64, 128])


In [ ]:
residual = embeddings + output

In [ ]:
layer_norm = nn.LayerNorm(embedding_dim)
normalized = layer_norm(residual)
print(normalized.shape)

torch.Size([16, 64, 128])


In [ ]:
ffn_linear1 = nn.Linear(embedding_dim, 4 * embedding_dim)
ffn_hidden = ffn_linear1(normalized)
print(ffn_hidden.shape)

torch.Size([16, 64, 512])


In [ ]:
activation = nn.GELU()
ffn_activated = activation(ffn_hidden)
print(ffn_activated.shape)

torch.Size([16, 64, 512])


In [ ]:
ffn_linear2 = nn.Linear(4 * embedding_dim, embedding_dim)
ffn_output = ffn_linear2(ffn_activated)
print(ffn_output.shape)

torch.Size([16, 64, 128])


In [ ]:
ffn_residual = normalized + ffn_output
print(ffn_residual.shape)

torch.Size([16, 64, 128])


In [ ]:
layer_norm2 = nn.LayerNorm(embedding_dim)
output = layer_norm2(ffn_residual)
print(output.shape)

torch.Size([16, 64, 128])


In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.query_layer = nn.Linear(embedding_dim, embedding_dim)
        self.key_layer = nn.Linear(embedding_dim, embedding_dim)
        self.value_layer = nn.Linear(embedding_dim, embedding_dim)

        self.output_projection = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):

        batch_size, block_size, embedding_dim = x.shape

        Q = self.query_layer(x)
        K = self.key_layer(x)
        V = self.value_layer(x)

        Q = Q.view(batch_size, block_size, self.num_heads, self.head_dim)
        K = K.view(batch_size, block_size, self.num_heads, self.head_dim)
        V = V.view(batch_size, block_size, self.num_heads, self.head_dim)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / (self.head_dim ** 0.5)

        mask = torch.tril(
            torch.ones(block_size, block_size, device=x.device)
        )

        scores = scores.masked_fill(mask == 0, float('-inf'))
        attention_weights = torch.softmax(scores, dim=-1)
        context = attention_weights @ V

        context = context.transpose(1, 2)

        context = context.contiguous().view(
            batch_size,
            block_size,
            embedding_dim
        )


        output = self.output_projection(context)

        return output

In [ ]:
attention = MultiHeadAttention(
    embedding_dim=embedding_dim,
    num_heads=num_heads
)

attention_output = attention(embeddings)

print("Attention output:", attention_output.shape)

Attention output: torch.Size([16, 64, 128])


In [ ]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
ffn = FeedForward(embedding_dim)
ffn_output = ffn(attention_output)
print("FFN output:", ffn_output.shape)

FFN output: torch.Size([16, 64, 128])


In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads):
        super().__init__()

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads
        )

        self.layer_norm1 = nn.LayerNorm(embedding_dim)

        self.ffn = FeedForward(embedding_dim)

        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):

        attention_output = self.attention(x)
        x = self.layer_norm1(x + attention_output)

        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + ffn_output)

        return x

In [ ]:
transformer_block = TransformerBlock(
    embedding_dim=embedding_dim,
    num_heads=num_heads
)

block_output = transformer_block(embeddings)

print("Transformer Block output:", block_output.shape)

Transformer Block output: torch.Size([16, 64, 128])


In [ ]:
class Transformer(nn.Module):

    def __init__(self, embedding_dim, num_heads, num_layers):
        super().__init__()

        self.blocks = nn.ModuleList([
            TransformerBlock(
                embedding_dim,
                num_heads
            )
            for _ in range(num_layers)
        ])

    def forward(self, x):

        for block in self.blocks:
            x = block(x)

        return x

In [ ]:
num_layers = 4

transformer = Transformer(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers
)

transformer_output = transformer(embeddings)

print("Transformer output:", transformer_output.shape)

Transformer output: torch.Size([16, 64, 128])


In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        block_size
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.position_embedding = nn.Embedding(
            block_size,
            embedding_dim
        )

        self.transformer = Transformer(
            embedding_dim,
            num_heads,
            num_layers
        )

        self.final_layer_norm = nn.LayerNorm(
            embedding_dim
        )

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, x):

        batch_size, block_size = x.shape

        token_emb = self.token_embedding(x)

        positions = torch.arange(
            block_size,
            device=x.device
        )

        pos_emb = self.position_embedding(positions)

        x = token_emb + pos_emb
        x = self.transformer(x)
        x = self.final_layer_norm(x)

        logits = self.lm_head(x)

        return logits


In [ ]:
model = GPTLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    block_size=block_size
).to(device)

x, y = get_batch(train_tokens, batch_size, block_size)
x = x.to(device)
y = y.to(device)

logits = model(x)

print("Input shape:", x.shape)
print("Logits shape:", logits.shape)

Input shape: torch.Size([16, 64])
Logits shape: torch.Size([16, 64, 50257])


In [ ]:
loss_function = nn.CrossEntropyLoss()

logits_flat = logits.view(
    batch_size * block_size,
    vocab_size
)

targets_flat = y.view(
    batch_size * block_size
)

loss = loss_function(
    logits_flat,
    targets_flat
)

print("Loss:", loss.item())

Loss: 10.997766494750977


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

print("Optimizer created")

Optimizer created


In [ ]:
optimizer.zero_grad()
logits = model(x)

logits_flat = logits.view(
    batch_size * block_size,
    vocab_size
)

targets_flat = y.view(
    batch_size * block_size
)

loss = loss_function(
    logits_flat,
    targets_flat
)

loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 10.997766494750977


In [ ]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          torch.cuda.get_device_properties(0).total_memory / 1024**3,
          "GB")

GPU available: True
GPU: Tesla T4
GPU memory: 14.56317138671875 GB


In [ ]:
num_steps = 1000

for step in range(num_steps):

    x, y = get_batch(
        train_tokens,
        batch_size,
        block_size
    )

    x = x.to(device)
    y = y.to(device)
    optimizer.zero_grad()

    logits = model(x)

    logits_flat = logits.view(
        batch_size * block_size,
        vocab_size
    )

    targets_flat = y.view(
        batch_size * block_size
    )

    loss = loss_function(
        logits_flat,
        targets_flat
    )

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(
            f"Step {step}: Loss {loss.item():.4f}"
        )

Step 0: Loss 10.8833
Step 100: Loss 6.5694
Step 200: Loss 5.9206
Step 300: Loss 5.7715
Step 400: Loss 5.6182
Step 500: Loss 5.2599
Step 600: Loss 5.2192
Step 700: Loss 5.4503
Step 800: Loss 4.7279
Step 900: Loss 4.8486


In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens):


    tokens = tokenizer.encode(prompt)
    x = torch.tensor(
        tokens,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)


    for _ in range(max_new_tokens):


        x_cond = x[:, -block_size:]
        logits = model(x_cond)

        logits = logits[:, -1, :]

        probabilities = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.argmax(
            probabilities,
            dim=-1,
            keepdim=True
        )

        x = torch.cat(
            (x, next_token),
            dim=1
        )

    generated_text = tokenizer.decode(
        x[0].tolist()
    )

    return generated_text

In [ ]:
prompt = "ROMEO:"

generated_text = generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=200
)

print(generated_text)

ROMEO:
I'll be a man, and the king,
And I'll be a man, and the king,
And I'll be the king, and the king.

KING RICHARD III:
I'll be a man, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And I'll be the king, and the king,
And
